<div class="blog-language-switch" role="group" aria-label="Article language"><span aria-current="page">English</span><a href="/ipynb/zh-CN/Computer-Science/Computer-Organization/11-parallelism-and-multicore.html" lang="zh-CN" hreflang="zh-CN">中文</a></div>

[Back to Computer Organization and Architecture guideline](Computer-Organization.html)

## **Parallelism and Multicore Systems** {#parallelism-and-multicore-systems}

Chapter 10 followed one I/O request through control, data transfer, and completion. A modern machine rarely handles only one request or one instruction stream. Several instructions may overlap inside a core, one instruction may operate on many values, several hardware threads may share a core, and many cores or accelerators may execute concurrently. This chapter explains where that parallel work comes from and what new correctness obligations appear when processing elements communicate.

**Concurrency** means that multiple activities are in progress during overlapping periods; **parallelism** means that at least some of them execute at the same instant on different resources. A single-core operating system can provide concurrency by time-slicing, while a multicore processor can provide both concurrency and physical parallelism. The distinction matters because concurrency creates interleavings and synchronization requirements even when no true simultaneous execution occurs.

The chapter follows a progression: first identify independent work, then map it to execution resources, then make communication correct through coherence, consistency, and synchronization, and finally measure whether the extra resources actually improve performance.

### **Why Hardware Uses Parallelism** {#why-hardware-uses-parallelism}

For many years, higher clock frequency and deeper pipelines made a single instruction stream run faster. That approach encountered several limits. Higher frequency raises power and heat; deeper speculation has diminishing returns when dependencies and branch mispredictions restrict useful work; and memory latency has not fallen at the same rate as arithmetic latency. Parallel hardware therefore spends additional transistors on more execution lanes, hardware-thread contexts, cores, and specialized engines instead of relying only on a faster clock.

Parallelism can target two different goals:

- **latency reduction** finishes one job sooner by dividing its critical work;
- **throughput improvement** completes more independent jobs per unit time, even if one job is not much faster.

If a workload takes $T_1$ seconds on one processing resource and $T_P$ seconds on $P$ resources, its measured speedup is

$$
S(P) = \frac{T_1}{T_P}.
$$

Here, $P$ is the number of processing resources and $S(P)$ is the factor by which execution becomes faster. A speedup of $4$ means the parallel execution takes one quarter of the original time. It does **not** follow that four resources guarantee $S(4)=4$: serial work, communication, synchronization, resource contention, and load imbalance all increase $T_P$.

| Goal | Useful workload shape | Typical mechanism | Main limitation |
|---|---|---|---|
| reduce one task's latency | task can be split with a short critical path | ILP, SIMD, multicore decomposition | dependencies and serial fraction |
| increase throughput | many independent requests or elements | SMT, multiple cores, GPU lanes | bandwidth and shared-resource contention |
| hide long latency | another instruction or thread is ready during a stall | out-of-order execution, hardware multithreading | insufficient independent work |
| improve energy efficiency | many simple operations replace a few complex speculative ones | vectors, GPUs, accelerators | narrower applicability |

A design is successful only when the software exposes work at the granularity that the hardware can exploit. Adding cores to a sequential dependency chain adds capacity but does not shorten that chain.

### **Forms of Parallelism** {#forms-of-parallelism}

Parallelism is easier to reason about when classified by granularity. Instruction-level parallelism overlaps operations from one instruction stream, data-level parallelism applies similar operations across many values, and thread-level parallelism runs independent instruction streams. Real processors combine all three.

::: {.diagram-scroll .wide-diagram}
![Instruction-level, data-level, and thread-level parallelism expose independent work at progressively larger granularities.](assets/parallelism-levels.svg)
:::

| Form | Unit of independent work | Usually discovered by | Representative hardware |
|---|---|---|---|
| ILP | instructions within one thread | compiler and dynamic scheduler | pipeline, superscalar issue, out-of-order core |
| DLP | elements of arrays, vectors, or tensors | programmer, compiler, or library | SIMD lanes, vector units, GPU warps |
| TLP | threads, tasks, or requests | programmer, runtime, and operating system | SMT contexts, cores, sockets |

#### **Instruction-Level Parallelism** {#instruction-level-parallelism}

Instruction-level parallelism exists when instructions from one thread do not all depend on one another and can therefore overlap. Chapter 07 introduced pipelining and hazards; ILP extends that idea with multiple-issue execution, register renaming, branch prediction, speculation, and out-of-order scheduling. The architectural state still appears to follow program order even though micro-operations may execute in a different order internally.

The scheduler must respect several constraints:

- a **read-after-write (RAW)** dependency is a true data dependency and cannot be renamed away;
- **write-after-read (WAR)** and **write-after-write (WAW)** name dependencies can often be removed by register renaming;
- a control dependency may require branch prediction and recovery;
- uncertain memory aliases can prevent a load from safely passing an older store;
- finite issue ports, functional units, queues, and cache bandwidth create structural limits.

For a dependency graph, the **critical path** is the longest latency-weighted chain from an input operation to a final result. Even with unlimited execution units, execution cannot finish sooner than this path. If all operations took $T_{serial}$ cycles sequentially and the critical path took $T_{critical}$ cycles, then $T_{serial}/T_{critical}$ is an optimistic ILP upper bound before resource conflicts and prediction failures.

<details>
<summary>Python model: compute an instruction dependency critical path</summary>

```python
# Operations are listed in topological order. Latency is measured in cycles.
operations = {
    'load_a': {'latency': 3, 'depends_on': []},
    'load_b': {'latency': 3, 'depends_on': []},
    'multiply': {'latency': 2, 'depends_on': ['load_a', 'load_b']},
    'load_c': {'latency': 3, 'depends_on': []},
    'add': {'latency': 1, 'depends_on': ['multiply', 'load_c']},
    'store': {'latency': 1, 'depends_on': ['add']},
}

finish_cycle = {}
for name, operation in operations.items():
    # An operation becomes ready only after every true dependency finishes.
    ready_cycle = max(
        (finish_cycle[parent] for parent in operation['depends_on']),
        default=0,
    )
    finish_cycle[name] = ready_cycle + operation['latency']

serial_cycles = sum(op['latency'] for op in operations.values())
critical_path_cycles = max(finish_cycle.values())
optimistic_ilp = serial_cycles / critical_path_cycles

print(f'serial={serial_cycles}, critical path={critical_path_cycles}')
print(f'optimistic ILP upper bound={optimistic_ilp:.2f}x')
assert (serial_cycles, critical_path_cycles) == (13, 7)
```

</details>

This model assumes unlimited functional units and perfect knowledge, so it is a lower bound on time rather than a cycle-accurate processor simulation. Its value is conceptual: independent loads overlap, but the multiply-add-store chain remains sequential.

#### **Data-Level Parallelism** {#data-level-parallelism}

Data-level parallelism appears when the same or similar operation can be applied independently to many data elements. Adding two arrays, filtering pixels, comparing database keys, and multiplying tensor tiles are natural examples. DLP is attractive because one decoded instruction or one control schedule can keep many arithmetic lanes busy.

For $N$ independent elements and hardware width $W$, an ideal lane machine needs approximately

$$
N_{groups} = \left\lceil \frac{N}{W} \right\rceil
$$

groups of work. $N$ is the total element count, $W$ is the number of elements processed per group, and the ceiling accounts for a final partial group. This is not automatically a $W$-fold speedup: loads, stores, shuffles, reductions, masks, alignment, and memory bandwidth may dominate.

DLP can be expressed through fixed-width SIMD instructions, vector-length-agnostic instructions, GPU threads, or high-level tensor operations. The essential property is element independence, not a particular instruction set. Reductions such as a sum are only partly independent: elements can be combined in parallel as a tree, but intermediate results must eventually merge.

#### **Thread-Level Parallelism** {#thread-level-parallelism}

Thread-level parallelism uses multiple independent instruction streams. Each thread has its own program counter, register state, and control flow. Threads may process unrelated requests, different partitions of one dataset, or pipeline stages. Hardware threads can share an execution core, while software threads can be scheduled across different cores and sockets.

The main challenge is no longer finding independent instructions inside one stream. It is dividing work so that threads receive comparable amounts, minimizing communication, and defining when shared state may be read or changed. A common pattern gives each thread a private partition and combines only compact partial results.

<details>
<summary>Python model: partition work and combine thread-local reductions</summary>

```python
def partition_round_robin(values, workers):
    # Each worker receives a private list, so the main loop needs no shared writes.
    partitions = [[] for _ in range(workers)]
    for index, value in enumerate(values):
        partitions[index % workers].append(value)
    return partitions

def parallel_sum_model(values, workers):
    partitions = partition_round_robin(values, workers)
    # These local sums represent independent thread work.
    partial_sums = [sum(partition) for partition in partitions]
    # Only the compact partial results are combined at the synchronization point.
    return sum(partial_sums), partitions, partial_sums

values = list(range(1, 17))
total, partitions, partial_sums = parallel_sum_model(values, workers=4)
print(partitions)
print(partial_sums, total)
assert total == sum(values) == 136
```

</details>

The code models decomposition rather than claiming that Python list operations execute in parallel. In a real runtime, partition size must be large enough that useful work exceeds task creation, scheduling, synchronization, and data-movement overhead.

### **Flynn's Taxonomy** {#flynns-taxonomy}

Flynn's taxonomy classifies a computational model by the number of concurrent **instruction streams** and **data streams**. It is a vocabulary for reasoning about organization, not a complete description of a commercial processor. A modern multicore CPU is normally MIMD at the core level while each core also executes SIMD instructions; one machine can therefore contain several categories at different levels.

::: {.diagram-scroll .wide-diagram}
![Flynn's taxonomy places SISD, SIMD, MISD, and MIMD in a matrix of instruction-stream and data-stream multiplicity.](assets/flynn-taxonomy-overview.svg)
:::

| Category | Meaning | Typical interpretation | Important caution |
|---|---|---|---|
| SISD | single instruction, single data | scalar sequential machine | a pipelined core can still exploit ILP internally |
| SIMD | single instruction, multiple data | packed SIMD or vector operation | lane masking reduces effective utilization |
| MISD | multiple instructions, single data | redundant fault-tolerant processing or specialized pipelines | uncommon as a general-purpose architecture |
| MIMD | multiple instructions, multiple data | multicore, multiprocessor, or cluster | communication may use shared memory or messages |

SIMT, the programming model used by many GPUs, resembles SIMD because a warp issues a common instruction across lanes, but it exposes per-thread registers and permits divergent control flow through masks. Calling a GPU simply SIMD hides these programming-model differences; calling the whole GPU MIMD hides the lockstep behavior within a warp. The useful description states the level being discussed.

### **SIMD and Vector Processing** {#simd-and-vector-processing}

A SIMD instruction names packed operands whose elements occupy fixed-width lanes, then applies one operation to all active lanes. For example, a 256-bit register may hold eight 32-bit integers. One packed addition can produce eight results, although the processor still needs enough load/store bandwidth and arithmetic resources to sustain that rate.

::: {.diagram-scroll}
![A single SIMD instruction is distributed to multiple processing units operating on different data elements.](assets/source-simd.svg)
:::

*Image source: [Cburnett, SIMD.svg, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:SIMD.svg), CC BY-SA 3.0.*

A **vector architecture** describes operations over a configurable number of elements. Implementations may process the architectural vector in several physical chunks, allowing the same binary to use different microarchitectural widths. The RISC-V Vector extension, for example, uses vector-length configuration and masks so software can strip-mine arrays without assuming one fixed physical lane count. Fixed-width SIMD and vector-length-agnostic designs share the DLP idea but expose different contracts to software.

Masks solve boundary conditions and conditional work. If $A$ of $W$ lanes are active for an instruction, its lane utilization is

$$
U_{lane} = \frac{A}{W}.
$$

$A$ is the number of lanes producing useful results and $W$ is the available lane count. A final three-element tail on an eight-lane unit has $U_{lane}=3/8=37.5\%$ for that instruction. Masking preserves correctness, but inactive lanes still represent unused capacity.

<details>
<summary>Python model: masked SIMD execution for a SAXPY-style operation</summary>

```python
def masked_saxpy(alpha, x, y, width):
    if len(x) != len(y):
        raise ValueError('x and y must have the same length')

    output = y.copy()
    trace = []
    for base in range(0, len(x), width):
        # The final group may contain fewer valid elements than physical lanes.
        active = min(width, len(x) - base)
        mask = [lane < active for lane in range(width)]
        for lane, enabled in enumerate(mask):
            if enabled:
                index = base + lane
                output[index] = alpha * x[index] + y[index]
        trace.append({'base': base, 'mask': mask, 'utilization': active / width})
    return output, trace

x = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
y = [10] * len(x)
output, trace = masked_saxpy(2, x, y, width=4)
print(output)
print(trace[-1])
assert output == [12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
assert trace[-1]['utilization'] == 0.5
```

</details>

| Mechanism | Control model | Width visible to software | Best fit | Common loss |
|---|---|---|---|---|
| scalar | one operation, one value | one element | irregular control and short work | low DLP |
| fixed-width SIMD | one packed instruction | fixed register width | dense loops and media processing | tails, shuffles, gather/scatter |
| vector-length-agnostic | one vector instruction | configurable active length | portable long-vector loops | setup and memory behavior |
| SIMT | per-thread program grouped into warps | warp width is a performance property | massive throughput workloads | divergence and uncoalesced memory |

### **Hardware Multithreading and Multicore Processors** {#hardware-multithreading-and-multicore-processors}

A hardware thread context contains enough architectural state to track an independent instruction stream, including a program counter, registers, privilege state, and selected control registers. Replicating this state is much cheaper than replicating an entire out-of-order core. Hardware multithreading uses that fact to keep shared execution resources busy when one thread stalls.

::: {.diagram-scroll .wide-diagram}
![Interleaved multithreading shares cycles, simultaneous multithreading shares issue slots, and multicore execution gives threads separate cores.](assets/hardware-threading-multicore.svg)
:::

- **Coarse-grained multithreading** switches after a long event such as a cache miss. It has low switching frequency but may leave short stalls uncovered.
- **Fine-grained multithreading** switches instruction streams frequently, often every cycle, to tolerate latency. One thread normally owns the issue opportunity in a cycle.
- **Simultaneous multithreading (SMT)** selects instructions from multiple threads in the same cycle to fill otherwise unused superscalar issue slots.
- **Multicore execution** replicates complete cores so threads can execute simultaneously with more independent resources, while still sharing some cache, memory, and interconnect capacity.

<details>
<summary>Python model: interleave ready instructions from stalled hardware threads</summary>

```python
from collections import deque

# Each tuple is (earliest ready cycle, instruction name).
queues = {
    'A': deque([(0, 'A0'), (5, 'A1'), (6, 'A2')]),  # A waits after a miss.
    'B': deque([(0, 'B0'), (1, 'B1'), (3, 'B2')]),
    'C': deque([(0, 'C0'), (2, 'C1'), (4, 'C2')]),
}

thread_names = list(queues)
last_choice = -1
schedule = []
for cycle in range(9):
    chosen = None
    # Round-robin among threads whose next instruction is ready.
    for offset in range(1, len(thread_names) + 1):
        candidate_index = (last_choice + offset) % len(thread_names)
        candidate = thread_names[candidate_index]
        if queues[candidate] and queues[candidate][0][0] <= cycle:
            chosen = candidate
            last_choice = candidate_index
            break

    if chosen is None:
        schedule.append((cycle, 'idle'))
    else:
        _, instruction = queues[chosen].popleft()
        schedule.append((cycle, instruction))

print(schedule)
assert all(instruction != 'idle' for _, instruction in schedule)
assert len(schedule) == 9
```

</details>

The example shows latency hiding rather than faster execution of thread A itself. Threads also compete for reorder-buffer entries, issue ports, cache capacity, translation structures, and memory bandwidth. SMT can increase total throughput while making one thread slower or less predictable. Shared microarchitectural state can also create side channels, so operating systems may restrict SMT for selected security domains.

| Organization | Replicated state | Shared resources | Primary benefit | Main cost |
|---|---|---|---|---|
| interleaved MT | thread architectural contexts | nearly whole core | tolerate stalls cheaply | one issuing thread per cycle |
| SMT | thread contexts and some queues | issue engine, execution units, caches | fill superscalar slots | contention and unpredictability |
| multicore | complete cores and private caches | LLC, interconnect, memory, I/O | true TLP capacity | coherence and communication |

### **Shared-Memory Multiprocessors** {#shared-memory-multiprocessors}

A shared-memory multiprocessor lets processors communicate by loading and storing a common address space. This abstraction is convenient: a pointer can name shared data, and ordinary load/store instructions transfer values. It is not physically simple. Private caches create replicas, memory controllers may be distributed across sockets, and synchronization must turn ordinary memory operations into well-defined communication.

::: {.diagram-scroll}
![A symmetric multiprocessor connects several processors to one shared memory through a bus or switch.](assets/source-shared-memory.svg)
:::

*Image source: [Khazadum and Stannered, Shared memory.svg, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Shared_memory.svg), CC BY-SA 3.0.*

In a **uniform memory access (UMA)** organization, processors observe broadly similar main-memory latency. In a **non-uniform memory access (NUMA)** organization, each socket or node has nearby memory controllers and can reach remote memory through an inter-socket fabric. The address space remains shared, but physical placement affects latency and bandwidth. Cache-coherent NUMA is often abbreviated **ccNUMA**.

::: {.diagram-scroll}
![A NUMA system gives each processor node local memory and connects nodes so remote memory remains addressable at higher cost.](assets/source-numa.svg)
:::

*Image source: [Moop2000, NUMA.svg, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:NUMA.svg), public domain.*

If a fraction $p$ of accesses reaches local memory with latency $L_{local}$ and the remaining fraction $1-p$ reaches remote memory with latency $L_{remote}$, a simple average is

$$
L_{avg} = pL_{local} + (1-p)L_{remote}.
$$

$p$ is the local-access probability, while $L_{local}$ and $L_{remote}$ are representative access latencies. Increasing locality raises $p$ and lowers the weighted average. The formula omits queueing and overlap, but it makes placement visible: a shared virtual address does not imply uniform physical cost.

<details>
<summary>Python model: estimate NUMA latency from local and remote accesses</summary>

```python
def numa_average_latency(accesses, thread_node, page_node, local_ns, remote_ns):
    total = 0
    local_count = 0
    for page in accesses:
        is_local = page_node[page] == thread_node
        local_count += int(is_local)
        total += local_ns if is_local else remote_ns
    return total / len(accesses), local_count / len(accesses)

page_node = {'A': 0, 'B': 0, 'C': 1, 'D': 1}
accesses = ['A', 'B', 'A', 'C', 'D', 'A', 'B', 'C']
average_ns, local_fraction = numa_average_latency(
    accesses, thread_node=0, page_node=page_node, local_ns=90, remote_ns=150
)
print(f'local fraction={local_fraction:.2f}, average={average_ns:.1f} ns')
assert local_fraction == 0.625
assert average_ns == 112.5
```

</details>

Operating systems use affinity and first-touch placement to keep threads near their pages. Parallel programs improve locality by partitioning data, binding worker threads, and avoiding frequent movement of writable cache lines across nodes.

| Property | UMA/SMP | ccNUMA |
|---|---|---|
| address space | shared | shared |
| memory latency | approximately uniform | local faster than remote |
| scaling | simpler at small processor counts | better capacity and bandwidth scaling |
| software concern | contention and cache behavior | all UMA concerns plus placement and affinity |

### **Cache Coherence** {#cache-coherence}

Private caches are essential for latency and bandwidth, but they create multiple physical copies of a shared memory block. **Cache coherence** provides a per-address contract that prevents processors from indefinitely using incompatible values. Informally, writes to one location must be serialized, and a read must eventually return the value of an appropriate write in that order.

::: {.diagram-scroll}
![A simple coherent system coordinates private caches and shared memory so processors do not continue using contradictory copies.](assets/source-cache-coherence.svg)
:::

*Image source: [Careless hx, Cache coherence.svg, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Cache_coherence.svg), CC BY-SA 4.0.*

A useful coherence invariant is **single writer or multiple readers (SWMR)**: at one logical time, either one cache has permission to modify a line or several caches may hold read-only copies. A second property is **data-value consistency**: when ownership moves, the new owner receives the newest value, whether from another cache, a shared last-level cache, or memory.

Most general-purpose coherent systems use **write invalidate**. Before writing a shared line, a core obtains exclusive ownership and invalidates peer copies. Write-update protocols instead send the new value to sharers; they can help repeatedly read shared data but may consume much more interconnect bandwidth.

#### **The Coherence Problem** {#the-coherence-problem}

Suppose core 0 and core 1 both cache a line containing `x=0`. Core 0 writes `x=1` only into its private write-back cache. Without a protocol, core 1 could continue reading zero forever and memory could also remain stale. Coherence transactions establish ownership, invalidate or downgrade replicas, and transfer dirty data when necessary.

The unit of coherence is normally a **cache line**, not an individual variable. This produces **false sharing** when threads write different variables located in the same line. The result remains correct because coherence works, but the line repeatedly transfers between cores even though the threads do not logically share a value.

::: {.diagram-scroll .wide-diagram}
![False sharing makes one cache line ping-pong between writers; padding or thread-private aggregation places independent counters in separate lines.](assets/false-sharing-cache-lines.svg)
:::

<details>
<summary>Python model: count ownership transfers caused by false sharing</summary>

```python
def ownership_transfers(writes, line_of_variable):
    owner_by_line = {}
    transfers = 0
    for core, variable in writes:
        line = line_of_variable[variable]
        previous_owner = owner_by_line.get(line)
        if previous_owner is not None and previous_owner != core:
            transfers += 1
        owner_by_line[line] = core
    return transfers

# Two cores repeatedly update independent counters in alternating order.
writes = [(0, 'A'), (1, 'B')] * 4
packed = {'A': 0, 'B': 0}   # Both variables occupy one cache line.
padded = {'A': 0, 'B': 1}   # Each variable occupies a separate line.

packed_transfers = ownership_transfers(writes, packed)
padded_transfers = ownership_transfers(writes, padded)
print(packed_transfers, padded_transfers)
assert packed_transfers == 7
assert padded_transfers == 0
```

</details>

Padding is not universally free because it increases memory footprint and can reduce cache density. Prefer thread-private accumulation followed by a reduction when the algorithm permits it; use padding when frequently written fields must remain independently shared.

#### **Snooping and Directory Protocols** {#snooping-and-directory-protocols}

A coherence request must discover which caches hold a line and whether one contains a newer dirty value. **Snooping** and **directory** protocols answer this discovery question differently.

::: {.diagram-scroll .wide-diagram}
![Snooping broadcasts each request to all caches, whereas a directory records sharers and sends targeted coherence messages.](assets/snooping-directory-comparison.svg)
:::

A snooping system places requests on a broadcast medium or another interconnect that supplies an observable order. Every cache checks, or snoops, each request against its tags. This is conceptually direct and works well for modest core counts, but broadcast bandwidth and electrical or logical fan-out become expensive as the system grows.

A directory associates metadata with each tracked line. The metadata identifies an owner or a set of sharers, allowing the home node to contact only relevant caches. A full bit-vector directory for $M$ memory lines and $P$ processors needs approximately

$$
B_{directory} = M \times P
$$

sharer bits before state and implementation overhead. $M$ is the number of tracked lines and $P$ is the processor count. Large systems therefore use sparse pointers, compressed vectors, hierarchical directories, or limited-pointer schemes rather than always storing one bit per processor per line.

<details>
<summary>Python model: compare broadcast observers with targeted invalidations</summary>

```python
def coherence_message_summary(processors, sharers, requester):
    targets = sorted(sharer for sharer in sharers if sharer != requester)
    return {
        # Every other cache observes a snooping request.
        'snoop_observers': processors - 1,
        # A directory sends invalidations only to known peer sharers.
        'directory_invalidation_targets': targets,
        'directory_target_count': len(targets),
    }

summary = coherence_message_summary(
    processors=64, sharers={2, 7, 19}, requester=2
)
print(summary)
assert summary['snoop_observers'] == 63
assert summary['directory_invalidation_targets'] == [7, 19]
```

</details>

| Property | Snooping | Directory |
|---|---|---|
| discovery | broadcast request | consult per-line metadata |
| ordering | often supplied by shared medium | protocol and network maintain required order |
| traffic | all caches observe requests | messages target owner and sharers |
| metadata | distributed in cache tags | explicit home/directory state |
| natural scale | small to moderate systems | many-core and multi-socket systems |
| complexity | broadcast arbitration | indirection, races, transient states |

#### **MESI Intuition** {#mesi-intuition}

MESI is a write-invalidate protocol named after four stable states for each cached line: **Modified**, **Exclusive**, **Shared**, and **Invalid**. The state records whether the local copy is usable, whether peers may also have a copy, and whether the local data is newer than the lower memory level.

::: {.diagram-scroll .wide-diagram}
![A simplified MESI diagram connects Invalid, Exclusive, Shared, and Modified states through processor reads, processor writes, snoop reads, and invalidations.](assets/mesi-simplified.svg)
:::

| State | Local read? | Local write without first invalidating peers? | Other cached copies? | Matches lower memory? |
|---|---:|---:|---:|---:|
| M: Modified | yes | yes | no | no |
| E: Exclusive | yes | yes, silently changes E to M | no | yes |
| S: Shared | yes | no, ownership must be acquired | possible | yes |
| I: Invalid | no | no, line or ownership must be acquired | irrelevant | no usable local copy |

A read miss can enter E when no other cache reports a copy, or S when another copy exists. A write to S requests an upgrade and invalidates sharers. A write to E can become M without an external transaction because exclusivity is already known. If another core reads a Modified line, the owner supplies or writes back the newest data and normally downgrades to a shared state.

<details>
<summary>Python model: trace common stable-state MESI transitions</summary>

```python
def processor_read(state, another_cache_has_copy):
    if state == 'I':
        return 'S' if another_cache_has_copy else 'E'
    return state

def processor_write(state):
    if state in {'E', 'M'}:
        return 'M'
    if state == 'S':
        return 'M'  # An upgrade invalidates peer sharers first.
    raise ValueError('I requires an ownership request before the write')

def snoop_read(state):
    # Another core requests a readable copy.
    return 'S' if state in {'M', 'E'} else state

def snoop_invalidate(state):
    return 'I' if state in {'M', 'E', 'S'} else state

state = 'I'
trace = [state]
state = processor_read(state, another_cache_has_copy=False)
trace.append(state)       # I -> E
state = processor_write(state)
trace.append(state)       # E -> M
state = snoop_read(state)
trace.append(state)       # M -> S
state = snoop_invalidate(state)
trace.append(state)       # S -> I
print(trace)
assert trace == ['I', 'E', 'M', 'S', 'I']
```

</details>

Real implementations include transient states for requests and acknowledgements in flight, races between local and snooped events, retries, and implementation-specific message names. MESI is therefore an intuition model, not a complete controller specification.

| Protocol | Stable states beyond invalid | Distinguishing idea |
|---|---|---|
| MSI | Modified, Shared | simpler; a private clean line is still Shared |
| MESI | Modified, Exclusive, Shared | E permits a silent first write |
| MOESI | adds Owned | dirty data may be shared while one cache remains responsible |
| MESIF | adds Forward | one clean sharer is selected to respond |

### **Memory Consistency** {#memory-consistency}

Coherence answers a per-location question: in what order are writes to `x` observed, and which value may a later read of `x` return? **Memory consistency** answers a broader question: what combinations of observations are legal when a program accesses several locations from several threads? A system can be perfectly coherent for every address and still permit an outcome that surprises a programmer who assumed all operations became visible in program order.

::: {.diagram-scroll .wide-diagram}
![In the store-buffering litmus test, each thread stores one variable and reads the other; weak visibility can let both loads see zero unless synchronization adds ordering.](assets/memory-consistency-litmus.svg)
:::

**Sequential consistency (SC)** gives a simple model: the result is as if all operations from all threads were interleaved in one total order that preserves each thread's program order. Real processors often implement weaker models so stores can buffer, loads can proceed around unrelated operations, and the memory system can overlap transactions. RISC-V's base model, RVWMO, is one example of a weak ordering model; explicit fences and acquire/release atomic operations add required ordering edges.

In the illustrated store-buffering test, thread 0 executes `x=1; r0=y` and thread 1 executes `y=1; r1=x`, with both variables initially zero. If each store remains temporarily private in its core's store buffer, both later loads can read zero from coherent memory. Each address still has a valid write order, so this outcome is about consistency rather than a coherence failure.

<details>
<summary>Python model: reproduce the store-buffering outcome</summary>

```python
memory = {'x': 0, 'y': 0}
store_buffers = {0: {}, 1: {}}

def buffered_store(thread, address, value):
    # The issuing thread records the store before global memory sees it.
    store_buffers[thread][address] = value

def load(thread, address):
    # A thread forwards its own pending store to the same address only.
    return store_buffers[thread].get(address, memory[address])

def flush(thread):
    memory.update(store_buffers[thread])
    store_buffers[thread].clear()

buffered_store(0, 'x', 1)
buffered_store(1, 'y', 1)
r0 = load(0, 'y')
r1 = load(1, 'x')
print('observed before flush:', r0, r1)
assert (r0, r1) == (0, 0)

flush(0)
flush(1)
assert memory == {'x': 1, 'y': 1}
```

</details>

Correct software should use the language's synchronization primitives rather than placing ad hoc hardware fences around ordinary data races. The compiler memory model, ISA memory model, and microarchitecture must agree. A release operation publishes earlier writes; a matching acquire prevents later reads from moving before the synchronization point. Together they can establish a **happens-before** relationship.

| Concept | Scope | Question answered | Mechanism |
|---|---|---|---|
| coherence | one cache line or address | which write to this location may this read observe? | coherence protocol |
| consistency | operations across addresses and processors | which global observations are legal? | architectural memory model |
| synchronization | selected program events | which operations must become ordered and visible? | atomics, locks, fences, barriers |
| mutual exclusion | critical section | who may update protected state now? | lock protocol built on atomics |

### **Hardware Support for Synchronization** {#hardware-support-for-synchronization}

A normal increment is a load, an arithmetic operation, and a store. Two threads can interleave those steps and lose an update. Hardware therefore provides **atomic read-modify-write (RMW)** operations whose read and write appear indivisible with respect to competing accesses. Software uses these primitives to build locks, counters, queues, barriers, reference counts, and lock-free data structures.

::: {.diagram-scroll .wide-diagram}
![A non-atomic increment loses an update, while atomic RMW and LR/SC make the state transition indivisible and retry interference.](assets/atomic-synchronization.svg)
:::

Common primitives include atomic swap, compare-and-swap (CAS), fetch-and-add, and load-reserved/store-conditional (LR/SC). CAS writes a new value only when memory still equals an expected value. LR/SC creates a reservation with a load and allows the conditional store to succeed only if no conflicting event invalidated that reservation. RISC-V's A extension provides LR/SC and atomic memory operations for inter-processor synchronization.

Atomicity alone is not sufficient. An operation also needs an ordering mode such as **relaxed**, **acquire**, **release**, **acquire-release**, or **sequentially consistent**, depending on what surrounding memory it must publish or observe. Stronger ordering is easier to reason about but can restrict compiler and hardware optimization.

<details>
<summary>Python model: a compare-and-swap retry after interference</summary>

```python
class AtomicIntegerModel:
    def __init__(self, value=0):
        self.value = value

    def load(self):
        return self.value

    def compare_exchange(self, expected, desired):
        if self.value != expected:
            return False
        self.value = desired
        return True

counter = AtomicIntegerModel(0)

# Thread A and thread B both read the old value.
observed_by_a = counter.load()
observed_by_b = counter.load()

# A wins the atomic transition from 0 to 1.
assert counter.compare_exchange(observed_by_a, observed_by_a + 1)

# B's stale expectation fails, so B reloads and retries.
first_attempt_by_b = counter.compare_exchange(observed_by_b, observed_by_b + 1)
assert not first_attempt_by_b
observed_by_b = counter.load()
assert counter.compare_exchange(observed_by_b, observed_by_b + 1)
print(counter.value)
assert counter.value == 2
```

</details>

The Python class is a state-machine model, not an implementation of a hardware atomic instruction. In production code, use atomics and synchronization supplied by the language and runtime. A handwritten check followed by a store is not atomic.

| Mechanism | What it guarantees | Waiting behavior | Suitable use | Main risk |
|---|---|---|---|---|
| atomic RMW | indivisible update of one atomic object | none by itself | counters and lock-free building blocks | cache-line contention |
| spin lock | one owner of critical section | repeatedly polls | very short hold time, dedicated cores | wastes cycles under long waits |
| blocking mutex | one owner with scheduler support | waiter sleeps or parks | longer or unpredictable waits | context-switch overhead |
| barrier | threads do not pass until participants arrive | spins or blocks | phase-based parallel algorithms | straggler determines progress |
| lock-free structure | system-wide progress without a lock | retries on conflict | high-concurrency specialized structures | subtle correctness and reclamation |

### **GPUs and Domain-Specific Accelerators** {#gpus-and-domain-specific-accelerators}

A CPU spends substantial area and energy reducing the latency of irregular instruction streams through large caches, branch prediction, speculative out-of-order execution, and sophisticated control. A GPU spends more of its resources on arithmetic lanes and hardware-thread state so it can sustain high throughput across many similar operations. Neither is universally better; they optimize different workload shapes.

GPU programming launches a **grid** of thread blocks. Threads within a block can cooperate through low-latency shared memory and barriers, while blocks should normally be independently schedulable. Hardware groups threads into warps or wavefronts for SIMT issue. The current NVIDIA CUDA programming model describes threads, blocks, grids, streaming multiprocessors, and 32-thread warps; those details should be treated as a programming contract and performance model rather than a promise about every internal circuit.

::: {.diagram-scroll}
![CUDA threads are organized into blocks, and blocks form a grid that can be distributed across streaming multiprocessors.](assets/source-cuda-thread-block.svg)
:::

*Image source: [NVIDIA, Block-thread.svg, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Block-thread.svg), CC BY 3.0.*

Two performance hazards follow from SIMT execution. **Branch divergence** makes one warp execute different paths with different active-lane masks. **Uncoalesced memory access** turns lane requests into many memory transactions. Regular control and adjacent addresses therefore matter as much as raw operation count.

::: {.diagram-scroll .wide-diagram}
![Within a SIMT warp, branch divergence masks lanes and scattered addresses require more memory transactions than coalesced accesses.](assets/gpu-simt-divergence.svg)
:::

<details>
<summary>Python model: map threads to blocks and estimate branch-lane utilization</summary>

```python
from math import ceil

def launch_geometry(elements, block_size):
    blocks = ceil(elements / block_size)
    active_threads = []
    for block in range(blocks):
        for thread in range(block_size):
            global_index = block * block_size + thread
            if global_index < elements:
                active_threads.append((block, thread, global_index))
    return blocks, active_threads

def branch_utilization(indices, predicate):
    path_a = sum(bool(predicate(index)) for index in indices)
    path_b = len(indices) - path_a
    # If both paths execute serially, useful lane work is spread over both passes.
    passes = int(path_a > 0) + int(path_b > 0)
    return len(indices) / (len(indices) * passes), (path_a, path_b)

blocks, threads = launch_geometry(elements=10, block_size=4)
utilization, split = branch_utilization(range(8), lambda index: index % 2 == 0)
print(f'blocks={blocks}, final thread={threads[-1]}')
print(f'branch split={split}, lane utilization={utilization:.0%}')
assert blocks == 3
assert threads[-1] == (2, 1, 9)
assert utilization == 0.5
```

</details>

A **domain-specific accelerator** narrows flexibility further. Tensor processors, video codecs, cryptographic engines, and network accelerators use specialized datapaths, local memories, dataflow schedules, and reduced-precision arithmetic to improve performance per watt. Their challenge is keeping the specialized units supplied with data and ensuring enough real applications map to the supported operations.

| Property | General-purpose CPU | GPU | Domain-specific accelerator |
|---|---|---|---|
| priority | low latency and flexible control | throughput across many threads | efficiency for a narrow operation family |
| control flow | strong irregular-control support | best when warps follow similar paths | often scheduled or highly constrained |
| parallel unit | instruction and thread | SIMT lane, warp, block | tile, systolic cell, codec stage, or engine |
| memory | large coherent caches | explicit hierarchy and high bandwidth | scratchpads, streams, and custom buffers |
| main risk | limited throughput per watt | divergence and data movement | poor utilization outside target domain |

### **Scalability and Parallel Speedup** {#scalability-and-parallel-speedup}

Scalability asks whether additional resources continue producing useful performance. **Strong scaling** keeps the total problem size fixed while increasing processors, so each processor receives less work. **Weak scaling** increases the problem size with the processor count so work per processor remains roughly constant. A design may weak-scale well while strong scaling saturates quickly.

Amdahl's law separates a workload into a parallelizable fraction $f$ and a serial fraction $1-f$. With $P$ processors, the idealized speedup is

$$
S_{Amdahl}(P) = \frac{1}{(1-f) + \frac{f}{P}}.
$$

$f$ is the fraction of one-processor execution time that can be parallelized, $1-f$ is the serial fraction, and $P$ is the number of processors. As $P$ approaches infinity, $f/P$ approaches zero, so the maximum speedup approaches $1/(1-f)$. If five percent is inherently serial, the limit is $1/0.05=20$, regardless of processor count.

::: {.diagram-scroll}
![Amdahl's law curves flatten as processor count increases, with the serial fraction imposing a finite maximum speedup.](assets/source-amdahl-law.svg)
:::

*Image source: [Daniels220 and contributors, AmdahlsLaw.svg, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:AmdahlsLaw.svg), CC BY-SA 3.0.*

Gustafson's law asks a different question: if the parallel machine allows a larger problem in the same elapsed time, how much scaled work is completed? Using serial fraction $\alpha$ measured in the scaled parallel run,

$$
S_{Gustafson}(P) = P - \alpha(P-1).
$$

$P$ is again the processor count and $\alpha$ is the observed serial-time fraction. Amdahl emphasizes fixed-size strong scaling; Gustafson emphasizes scaled problems and weak-scaling intuition. They answer different workload questions rather than contradicting one another.

A practical runtime model must also include overhead:

$$
T_P = T_{serial} + \frac{T_{parallel}}{P} + T_{communication} + T_{synchronization} + T_{imbalance}.
$$

$T_{serial}$ cannot be divided, $T_{parallel}/P$ is the ideal divided work, $T_{communication}$ moves data or cache lines, $T_{synchronization}$ covers waiting and atomic coordination, and $T_{imbalance}$ is time lost because some workers finish later than others. These overhead terms often grow with $P$, so real speedup can peak and then decline.

<details>
<summary>Python example: compare Amdahl and Gustafson speedup</summary>

```python
from math import isclose

def amdahl_speedup(parallel_fraction, processors):
    serial_fraction = 1 - parallel_fraction
    return 1 / (serial_fraction + parallel_fraction / processors)

def gustafson_speedup(serial_fraction, processors):
    return processors - serial_fraction * (processors - 1)

parallel_fraction = 0.95
serial_fraction = 0.05
for processors in [1, 2, 4, 8, 16, 64]:
    fixed = amdahl_speedup(parallel_fraction, processors)
    scaled = gustafson_speedup(serial_fraction, processors)
    efficiency = fixed / processors
    print(
        f'P={processors:>2}: Amdahl={fixed:>5.2f}x, '
        f'efficiency={efficiency:>6.1%}, Gustafson={scaled:>5.2f}x'
    )

assert isclose(amdahl_speedup(0.95, 64), 1280 / 83)
assert isclose(gustafson_speedup(0.05, 64), 60.85)
assert isclose(1 / (1 - parallel_fraction), 20.0)
```

</details>

Parallel efficiency is $E(P)=S(P)/P$. It reports how much of the ideal $P$-fold capacity becomes speedup. Falling efficiency is not automatically a failure: a lower-efficiency run may still finish much sooner, and weak scaling may be the actual goal. The metric must match the workload and service objective.

| Scaling symptom | Likely architectural cause | Useful investigation |
|---|---|---|
| speedup flattens at small $P$ | serial critical section or dependency chain | profile serial time and lock hold time |
| throughput rises but per-thread latency worsens | SMT or cache contention | compare isolated and co-scheduled counters |
| remote-socket slowdown | poor NUMA placement | inspect affinity, page placement, and remote traffic |
| cache misses rise with core count | capacity pressure or false sharing | measure LLC misses and ownership transfers |
| workers wait at barriers | load imbalance or stragglers | inspect per-worker completion time |
| GPU lanes underutilized | divergence, small launch, or irregular work | inspect active lanes, occupancy, and branch split |
| bandwidth saturates | memory or interconnect ceiling | measure bytes moved per unit work |

**Chapter summary.** Parallel performance begins with independent work at the instruction, data, or thread level. Flynn's taxonomy names stream organization, while SIMD, vector units, hardware multithreading, multicore CPUs, GPUs, and accelerators map different workload shapes onto hardware. Shared memory simplifies communication but private caches require coherence. Snooping broadcasts discovery; directories track sharers; MESI encodes common ownership states. Coherence orders one location, memory consistency constrains observations across locations, and atomic operations plus acquire/release ordering build synchronization. Finally, core or lane count is only capacity: serial work, communication, contention, imbalance, bandwidth, and data placement determine realized speedup. Chapter 12 will evaluate those tradeoffs together with power, reliability, and security at the complete-system level.